# Darknet Traffic Detection Project


In [1]:
# Import required libraries
import pandas as pd
import numpy as np

## Phase 1 — Dataset Acquisition

In [2]:
# Load the dataset from Excel
# Make sure Darknet.xlsx is in the same folder as this notebook
df = pd.read_excel("Darknet.xlsx")

# Remove empty columns if any exist
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))


Dataset shape: (141531, 85)
Number of columns: 85


In [3]:
# View first rows
df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Label.1
0,10.152.152.11-216.58.220.99-57158-443-6,10.152.152.11,57158,216.58.220.99,443,6,24/07/2015 04:09:48 PM,229,1,1,...,0,0,0,0,0,0.000000e+00,0,0.000000e+00,Non-Tor,AUDIO-STREAMING
1,10.152.152.11-216.58.220.99-57159-443-6,10.152.152.11,57159,216.58.220.99,443,6,24/07/2015 04:09:48 PM,407,1,1,...,0,0,0,0,0,0.000000e+00,0,0.000000e+00,Non-Tor,AUDIO-STREAMING
2,10.152.152.11-216.58.220.99-57160-443-6,10.152.152.11,57160,216.58.220.99,443,6,24/07/2015 04:09:48 PM,431,1,1,...,0,0,0,0,0,0.000000e+00,0,0.000000e+00,Non-Tor,AUDIO-STREAMING
3,10.152.152.11-74.125.136.120-49134-443-6,10.152.152.11,49134,74.125.136.120,443,6,24/07/2015 04:09:48 PM,359,1,1,...,0,0,0,0,0,0.000000e+00,0,0.000000e+00,Non-Tor,AUDIO-STREAMING
4,10.152.152.11-173.194.65.127-34697-19305-6,10.152.152.11,34697,173.194.65.127,19305,6,24/07/2015 04:09:45 PM,10778451,591,400,...,0,0,0,0,1437764990528710,3.117718e+06,1437764995910100,1.437765e+15,Non-Tor,AUDIO-STREAMING


In [4]:
# Check dataset information (column types, null counts)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141531 entries, 0 to 141530
Data columns (total 85 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Flow ID                     141531 non-null  object 
 1   Src IP                      141531 non-null  object 
 2   Src Port                    141531 non-null  int64  
 3   Dst IP                      141531 non-null  object 
 4   Dst Port                    141531 non-null  int64  
 5   Protocol                    141531 non-null  int64  
 6   Timestamp                   141531 non-null  object 
 7   Flow Duration               141531 non-null  int64  
 8   Total Fwd Packet            141531 non-null  int64  
 9   Total Bwd packets           141531 non-null  int64  
 10  Total Length of Fwd Packet  141531 non-null  int64  
 11  Total Length of Bwd Packet  141531 non-null  int64  
 12  Fwd Packet Length Max       141531 non-null  int64  
 13  Fwd Packet Len

## Phase 2 — Data Preprocessing

In [5]:

# Some columns were loaded as 'object' (text) but should be numbers.
# pd.to_numeric will convert them, and any bad values become NaN.
# ------------------------------------------------------------------
mixed_type_cols = [
    'Fwd Header Length',
    'Bwd Header Length',
    'Bwd Packets/s',
    'Packet Length Mean'
]

for col in mixed_type_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Mixed-type columns converted to numeric.")

Mixed-type columns converted to numeric.


In [6]:
# Replace infinite values with NaN, then drop all rows with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print('Dataset shape after cleaning NaN/Inf rows:', df.shape)

Dataset shape after cleaning NaN/Inf rows: (141481, 85)


In [7]:
# Remove socket-related columns AND Timestamp
# These are identifiers, not useful features for ML
columns_to_remove = [
    'Flow ID',
    'Src IP',
    'Dst IP',
    'Src Port',
    'Dst Port',
    'Timestamp'     
]

df.drop(columns=columns_to_remove, inplace=True, errors='ignore')

print('Dataset shape after removing socket/timestamp columns:', df.shape)
print('Remaining columns:', df.shape[1])

Dataset shape after removing socket/timestamp columns: (141481, 79)
Remaining columns: 79


## Phase 3 — Label Engineering

The dataset has two label columns:
- `Label`   → contains: Tor, Non-Tor, VPN, NonVPN  (this is for Case 2)
- `Label.1` → contains: traffic types like P2P, Browsing, etc.

**Case 1 (Binary):** We need to create a Benign vs Darknet label.
- "Non-Tor" maps to → **Benign**
- "Tor", "VPN", "NonVPN" map to → **Darknet**

**Case 2 (Multiclass):** Tor / Non-Tor / VPN / Non-VPN
- We use the `Label` column directly, but fix naming inconsistencies.

In [8]:

# Some rows have numeric values like '5494.505495' in the Label column
# These are invalid — keep only known valid labels
# ------------------------------------------------------------------
valid_labels = ['Non-Tor', 'Tor', 'VPN', 'NonVPN']
df = df[df['Label'].isin(valid_labels)]

print("Rows after removing corrupted label rows:", len(df))
print("\nLabel column unique values:", df['Label'].unique())

Rows after removing corrupted label rows: 141481

Label column unique values: ['Non-Tor' 'NonVPN' 'Tor' 'VPN']


In [9]:
# ------------------------------------------------------------------
# CASE 1: Binary Classification — Benign vs Darknet
# Map: Non-Tor → Benign | Tor, VPN, NonVPN → Darknet
# ------------------------------------------------------------------
label_map_binary = {
    'Non-Tor': 'Benign',
    'Tor':     'Darknet',
    'VPN':     'Darknet',
    'NonVPN':  'Darknet'
}

df['Label_Binary'] = df['Label'].map(label_map_binary)

print("Case 1 — Binary label distribution:")
print(df['Label_Binary'].value_counts())

Case 1 — Binary label distribution:
Label_Binary
Benign     93309
Darknet    48172
Name: count, dtype: int64


In [10]:
# ------------------------------------------------------------------
# CASE 2: Multiclass Classification — Tor / Non-Tor / VPN / Non-VPN
# The Label column already has these values, but 'NonVPN' should be
# standardised to 'Non-VPN' to match the paper
# ------------------------------------------------------------------
label_map_multi = {
    'Non-Tor': 'Non-Tor',
    'Tor':     'Tor',
    'VPN':     'VPN',
    'NonVPN':  'Non-VPN'   # standardise naming
}

df['Label_Multi'] = df['Label'].map(label_map_multi)

print("Case 2 — Multiclass label distribution:")
print(df['Label_Multi'].value_counts())

Case 2 — Multiclass label distribution:
Label_Multi
Non-Tor    93309
Non-VPN    23861
VPN        22919
Tor         1392
Name: count, dtype: int64


In [11]:
# ------------------------------------------------------------------
# Prepare X (features) and y (labels) for both cases
# Drop all label columns from features
# ------------------------------------------------------------------
label_cols = ['Label', 'Label.1', 'Label_Binary', 'Label_Multi']

X = df.drop(columns=label_cols, errors='ignore')
y_binary = df['Label_Binary']   # Case 1
y_multi  = df['Label_Multi']    # Case 2

print("Feature matrix shape (X):", X.shape)
print("Binary labels shape   (y_binary):", y_binary.shape)
print("Multi  labels shape   (y_multi):", y_multi.shape)

Feature matrix shape (X): (141481, 77)
Binary labels shape   (y_binary): (141481,)
Multi  labels shape   (y_multi): (141481,)


In [12]:
# Final sanity check — make sure X has only numeric columns
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print("WARNING: These columns are still non-numeric and will cause problems:", non_numeric)
else:
    print("All feature columns are numeric. Ready for Phase 4 (Train/Test Split)!")

print("\nFinal feature count:", X.shape[1])
print("Final row count:", X.shape[0])

All feature columns are numeric. Ready for Phase 4 (Train/Test Split)!

Final feature count: 77
Final row count: 141481


## Phase 4 — Train / Test Split

In [13]:

from sklearn.model_selection import train_test_split, StratifiedKFold

# ── Case 1: Binary (Benign vs Darknet) ──────────────────────────────
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X, y_binary,
    test_size=0.2,        # 20% goes to test
    random_state=42,      # makes results reproducible
    stratify=y_binary     # keeps class balance (important!)
)

# ── Case 2: Multiclass (Tor / Non-Tor / VPN / Non-VPN) ──────────────
X_train_mul, X_test_mul, y_train_mul, y_test_mul = train_test_split(
    X, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

# ── 5-Fold Cross-Validation setup (used later during training) ───────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Confirm shapes ────────────────────────────────────────────────────
print("=== Case 1 — Binary ===")
print(f"  Train: {X_train_bin.shape}  |  Test: {X_test_bin.shape}")
print(f"  Train labels:\n{y_train_bin.value_counts()}")
print(f"\n  Test labels:\n{y_test_bin.value_counts()}")

print("\n=== Case 2 — Multiclass ===")
print(f"  Train: {X_train_mul.shape}  |  Test: {X_test_mul.shape}")
print(f"  Train labels:\n{y_train_mul.value_counts()}")

=== Case 1 — Binary ===
  Train: (113184, 77)  |  Test: (28297, 77)
  Train labels:
Label_Binary
Benign     74647
Darknet    38537
Name: count, dtype: int64

  Test labels:
Label_Binary
Benign     18662
Darknet     9635
Name: count, dtype: int64

=== Case 2 — Multiclass ===
  Train: (113184, 77)  |  Test: (28297, 77)
  Train labels:
Label_Multi
Non-Tor    74647
Non-VPN    19089
VPN        18335
Tor         1113
Name: count, dtype: int64
